# Módulo 4: Aritmética Vectorizada, Funciones Universales y Broadcasting

## 🎯 Objetivos de Aprendizaje
Al finalizar este cuaderno, serás capaz de:
1. Ejecutar operaciones aritméticas vectorizadas (elemento a elemento y con escalares) eliminando la necesidad de bucles `for` en Python.
2. Aplicar las **tres reglas formales del Broadcasting** para operar arreglos de distintas dimensiones sin duplicar memoria física.
3. Utilizar funciones matemáticas universales (**`ufuncs`**) para cálculos de potencias, raíces, logaritmos, exponenciales y trigonometría sobre vectores y matrices completas.
4. Distinguir con precisión entre la multiplicación/división **elemento a elemento** (Hadamard) y las operaciones del álgebra lineal.

---

## 🗺️ Mapa Mental del Módulo

```text
                [ Computación Vectorizada en NumPy ]
                                  │
      ┌───────────────────────────┼───────────────────────────┐
      ▼                           ▼                           ▼
1. ARITMÉTICA BÁSICA       2. REGLAS DE BROADCASTING   3. FUNCIONES UNIVERSALES (ufuncs)
├── Escalares (+, -, *, /) ├── Alineación por la derecha ├── Potencias: square, power, sqrt
├── Elemento a elemento    ├── Dimensión 1 se estira     ├── Exp/Log: exp, log, log10
└── Producto Hadamard (*)  └── Error: ValueError shapes  └── Trig/Const: sin, cos, tan, pi

In [1]:
## Importación de numpy
import numpy as np

## 4.1 Aritmética Elemento a Elemento y Operaciones con Escalares

La **vectorización** permite aplicar operaciones matemáticas directamente sobre arreglos completos. Cuando operas dos arreglos del mismo `shape`, NumPy empareja cada celda `(i, j)` del primer arreglo con la celda `(i, j)` del segundo arreglo en código C precompilado.

### Operadores y Funciones Equivalentes
* **Suma:** `A + B` o `np.add(A, B)`
* **Resta:** `A - B` o `np.subtract(A, B)`
* **Multiplicación elemento a elemento (Producto de Hadamard):** `A * B` o `np.multiply(A, B)`
* **División elemento a elemento:** `A / B` o `np.divide(A, B)`
* **Potencia elemento a elemento:** `A ** n` o `np.power(A, n)`

### Sintaxis Mínima

In [10]:
# Vector 1D: Precios base ($) de 10 productos en catálogo
precios_base = np.array([15.0, 22.5, 10.0, 45.0, 30.0, 12.0, 18.5, 50.0, 25.0, 40.0])

# Operación con escalar: Aplicar un IVA del 15% a todos los productos simultáneamente
precios_con_iva = precios_base * 1.15
print("Precios base    :", precios_base)
print("Precios con IVA :", np.round(precios_con_iva, 2))

# Matriz A (3x4): Unidades vendidas en el Trimestre 1 (3 sucursales x 4 productos)
unidades_t1 = np.array([
    [120, 85,  200, 50],
    [95,  110, 180, 60],
    [150, 70,  220, 45]
])
print("\nTrimestre 1:\n", unidades_t1)

# Matriz B (3x4): Unidades vendidas en el Trimestre 2
unidades_t2 = np.array([
    [130, 90,  190, 55],
    [100, 115, 200, 75],
    [140, 80,  250, 50]
])
print("\nTrimestre 2:\n", unidades_t2)

# 1. Suma y resta elemento a elemento
total_semestral = unidades_t1 + unidades_t2
variacion = unidades_t2 - unidades_t1

print("\nTotal semestral (A + B):\n", total_semestral)
print("\nVariación T2 - T1 (B - A):\n", variacion)

# 2. División elemento a elemento: Tasa de crecimiento relativo ((T2 - T1) / T1)
crecimiento_pct = (variacion / unidades_t1) * 100
print("\nCrecimiento porcentual por celda (%):\n", np.round(crecimiento_pct, 1))

Precios base    : [15.  22.5 10.  45.  30.  12.  18.5 50.  25.  40. ]
Precios con IVA : [17.25 25.87 11.5  51.75 34.5  13.8  21.28 57.5  28.75 46.  ]

Trimestre 1:
 [[120  85 200  50]
 [ 95 110 180  60]
 [150  70 220  45]]

Trimestre 2:
 [[130  90 190  55]
 [100 115 200  75]
 [140  80 250  50]]

Total semestral (A + B):
 [[250 175 390 105]
 [195 225 380 135]
 [290 150 470  95]]

Variación T2 - T1 (B - A):
 [[ 10   5 -10   5]
 [  5   5  20  15]
 [-10  10  30   5]]

Crecimiento porcentual por celda (%):
 [[ 8.3  5.9 -5.  10. ]
 [ 5.3  4.5 11.1 25. ]
 [-6.7 14.3 13.6 11.1]]


:::{warning} Trampa Común: Confundir `A * B` con la Multiplicación Matricial
En NumPy, el operador `*` **siempre realiza una multiplicación celda por celda (Hadamard)**, incluso entre matrices 2D. No calcula filas por columnas. Para el producto matricial del álgebra lineal, se utiliza el operador `@` (que estudiaremos en el Módulo 7).
:::

:::{note} Sobre la "División" de Matrices y Vectores
En programación vectorizada, `A / B` divide cada elemento `A[i, j]` entre `B[i, j]`. Si algún elemento de `B` es `0`, NumPy **no detiene el programa** con una excepción; en su lugar, asigna `inf` (infinito) o `nan` (*Not a Number*) y emite una advertencia `RuntimeWarning`.
:::

### 🧩 Micro-reto 4.1
Dadas dos matrices de $3 \times 4$: `ingresos` y `costos` (creadas en la celda inferior):
1. Calcula la matriz `utilidad_bruta` restando `costos` de `ingresos`.
2. Calcula el `margen_porcentual` dividiendo `utilidad_bruta` entre `ingresos` y multiplicando el resultado por el escalar `100`. Redondea el resultado a 1 decimal usando `np.round(..., 1)`.

In [4]:
ingresos = np.array([
    [500, 620, 480, 710],
    [800, 750, 900, 820],
    [410, 450, 520, 490]
])

costos = np.array([
    [320, 400, 310, 450],
    [500, 480, 550, 510],
    [290, 300, 340, 320]
])

# Escribe tu solución aquí:

## 4.2 Principios y Reglas del Broadcasting

El **Broadcasting** (difusión) es el mecanismo que permite a NumPy realizar operaciones aritméticas entre arreglos de **diferentes formas (`shape`)** sin copiar datos en memoria. El arreglo más pequeño se "estira" virtualmente a lo largo de las dimensiones faltantes o unitarias para igualar al más grande.

### Las 3 Reglas Oficiales de Compatibilidad Dimensional
Cuando NumPy opera dos arreglos `A` y `B`, compara sus `shape` **desde la última dimensión (derecha) hacia la primera (izquierda)**:

1. **Relleno por la izquierda:** Si los arreglos tienen distinto `ndim`, el `shape` del arreglo con menos dimensiones se rellena con unos (`1`) a su izquierda.
   * *Ejemplo:* Matriz `(4, 3)` y Vector `(3,)` $\rightarrow$ el vector pasa a interpretarse como `(1, 3)`.
2. **Regla de estiramiento (`1` $\rightarrow$ `N`):** Dos dimensiones son compatibles si son iguales, o si **una de ellas es `1`**. La dimensión de tamaño `1` se estira virtualmente para coincidir con la otra.
3. **Incompatibilidad (`ValueError`):** Si en cualquier eje las dimensiones son distintas y ninguna es `1`, NumPy detiene la ejecución con `ValueError: operands could not be broadcast together`.

### Tabla de Diagnóstico Rápido

| Shape A | Shape B | Alineación por derecha | ¿Compatible? | Shape Resultante |
| :--- | :--- | :--- | :--- | :--- |
| `(4, 3)` | `(3,)` | `(4, 3)` vs `(1, 3)` | **Sí** (se repite por filas) | `(4, 3)` |
| `(4, 3)` | `(4, 1)` | `(4, 3)` vs `(4, 1)` | **Sí** (se repite por columnas) | `(4, 3)` |
| `(4, 1)` | `(1, 3)` | `(4, 1)` vs `(1, 3)` | **Sí** (malla exterior) | `(4, 3)` |
| `(4, 3)` | `(4,)` | `(4, 3)` vs `(1, 4)` | **No** (`3 != 4`) | `ValueError` |

### Sintaxis Mínima

In [14]:
# Matriz 4x3: Calificaciones base de 4 estudiantes (filas) en 3 exámenes (columnas)
notas = np.array([
    [7.0, 6.5, 8.0],
    [8.5, 7.5, 9.0],
    [6.0, 5.5, 7.0],
    [9.0, 8.5, 9.5]
])
print("Matriz notas:\n", notas)

# Caso 1: Sumar un bono distinto a cada EXAMEN (vector fila de shape (3,))
# Alineación: (4, 3) + (1, 3) -> Compatible
bono_por_examen = np.array([0.5, 1.0, 0.2])
print("\nVector bono:", bono_por_examen)
notas_con_bono = notas + bono_por_examen
print("\n1. Broadcasting por columnas (notas + bono_por_examen):\n", notas_con_bono)

# Caso 2: Restar una penalización distinta a cada ESTUDIANTE (vector columna de shape (4, 1))
# Alineación: (4, 3) - (4, 1) -> Compatible
penalizacion_estudiante = np.array([0.5, 0.0, 1.0, 0.2]).reshape(4, 1)
print("\nPenalización:", penalizacion_estudiante)
notas_ajustadas = notas - penalizacion_estudiante
print("\n2. Broadcasting por filas (notas - penalizacion_estudiante):\n", notas_ajustadas)

# Caso 3: Producto exterior mediante Broadcasting: (4, 1) * (1, 3) -> Matriz (4, 3)
col = np.array([10, 20, 30, 40])[:, np.newaxis]   # shape (4, 1)
fila = np.array([1, 2, 3])[np.newaxis, :]          # shape (1, 3)
tabla_multiplicar = col * fila
print("\n3. Malla generada por Broadcasting (4, 1) * (1, 3):\n", tabla_multiplicar)

Matriz notas:
 [[7.  6.5 8. ]
 [8.5 7.5 9. ]
 [6.  5.5 7. ]
 [9.  8.5 9.5]]

Vector bono: [0.5 1.  0.2]

1. Broadcasting por columnas (notas + bono_por_examen):
 [[7.5 7.5 8.2]
 [9.  8.5 9.2]
 [6.5 6.5 7.2]
 [9.5 9.5 9.7]]

Penalización: [[0.5]
 [0. ]
 [1. ]
 [0.2]]

2. Broadcasting por filas (notas - penalizacion_estudiante):
 [[6.5 6.  7.5]
 [8.5 7.5 9. ]
 [5.  4.5 6. ]
 [8.8 8.3 9.3]]

3. Malla generada por Broadcasting (4, 1) * (1, 3):
 [[ 10  20  30]
 [ 20  40  60]
 [ 30  60  90]
 [ 40  80 120]]


### 🧩 Micro-reto 4.2
Tienes una matriz `produccion` de $4 \times 5$ (4 fábricas $\times$ 5 días) generada con `np.full((4, 5), 100)`:
1. Multiplica cada **día** (columna) por un factor de demanda distinto usando el vector `demanda_diaria = np.array([1.0, 1.2, 1.5, 1.1, 0.8])`.
2. Al resultado anterior, súmale un incentivo fijo por **fábrica** (fila) definido en el vector `incentivo_fabrica = np.array([10, 20, 30, 40])`. *(Recuerda ajustar la forma de este vector para que el broadcasting funcione).*

In [6]:
produccion = np.full((4, 5), 100)
demanda_diaria = np.array([1.0, 1.2, 1.5, 1.1, 0.8])
incentivo_fabrica = np.array([10, 20, 30, 40])

# Escribe tu solución aquí:

## 4.3 Funciones Matemáticas Universales (`ufuncs`)

Una **Universal Function (`ufunc`)** es una función vectorizada que opera elemento por elemento sobre cualquier `ndarray` (1D, 2D o ND) devolviendo un arreglo de la misma forma.

### Catálogo Esencial de `ufuncs`
* **Potencias y Raíces:**
  * `np.square(x)`: Eleva cada elemento al cuadrado ($x^2$).
  * `np.power(x, p)`: Eleva cada elemento a la potencia `p` ($x^p$, donde `p` puede ser un escalar o una matriz de exponentes).
  * `np.sqrt(x)`: Calcula la raíz cuadrada ($\sqrt{x}$).
* **Exponenciales y Logaritmos:**
  * `np.exp(x)`: Calcula $e^x$.
  * `np.log(x)`: Logaritmo natural ($\ln(x)$, base $e$).
  * `np.log10(x)` y `np.log2(x)`: Logaritmos en base 10 y base 2.
* **Trigonometría y Constantes:**
  * `np.pi` y `np.e`: Constantes matemáticas de alta precisión.
  * `np.sin(x)`, `np.cos(x)`, `np.tan(x)`: Funciones trigonométricas (**operan en radianes**).
  * `np.deg2rad(grados)` y `np.rad2deg(radianes)`: Conversión angular vectorizada.

### Sintaxis Mínima

In [17]:
# 1. Trigonometría sobre un vector de ángulos notables en grados
grados = np.array([0, 30, 45, 60, 90, 120, 135, 150, 180, 270])
radianes = np.deg2rad(grados)

senos = np.round(np.sin(radianes), 3)
cosenos = np.round(np.cos(radianes), 3)
print("Ángulos (grados):", grados)
print("Seno            :", senos)
print("Coseno          :", cosenos)

# 2. Potencias y raíces sobre una matriz 3x3 de distancias (catetos)
catetos_a = np.array([[3, 5, 8], [7, 9, 12], [6, 20, 15]])
catetos_b = np.array([[4, 12, 15], [24, 12, 16], [8, 21, 20]])

print("\nCatetos A:\n", catetos_a)
print("\nCatetos B:\n", catetos_b)

# Teorema de Pitágoras vectorizado: c = sqrt(a^2 + b^2)
hipotenusas = np.sqrt(np.square(catetos_a) + np.square(catetos_b))
print("\nMatriz de hipotenusas calculadas vectorialmente:\n", hipotenusas)

# 3. Transformación logarítmica sobre datos con crecimiento exponencial (3x4)
poblacion_bacterias = np.array([
    [10, 100, 1000, 10000],
    [50, 500, 5000, 50000],
    [20, 200, 2000, 20000]
])

print("\nMatriz población de bacterias:\n", poblacion_bacterias)
log_base10 = np.log10(poblacion_bacterias)
print("\nEscala logarítmica (log10):\n", np.round(log_base10, 2))

Ángulos (grados): [  0  30  45  60  90 120 135 150 180 270]
Seno            : [ 0.     0.5    0.707  0.866  1.     0.866  0.707  0.5    0.    -1.   ]
Coseno          : [ 1.     0.866  0.707  0.5    0.    -0.5   -0.707 -0.866 -1.    -0.   ]

Catetos A:
 [[ 3  5  8]
 [ 7  9 12]
 [ 6 20 15]]

Catetos B:
 [[ 4 12 15]
 [24 12 16]
 [ 8 21 20]]

Matriz de hipotenusas calculadas vectorialmente:
 [[ 5. 13. 17.]
 [25. 15. 20.]
 [10. 29. 25.]]

Matriz población de bacterias:
 [[   10   100  1000 10000]
 [   50   500  5000 50000]
 [   20   200  2000 20000]]

Escala logarítmica (log10):
 [[1.  2.  3.  4. ]
 [1.7 2.7 3.7 4.7]
 [1.3 2.3 3.3 4.3]]


:::{warning} Trampa Común: Usar `math.sqrt()` o `math.sin()` de Python en lugar de `np.sqrt()`
Si intentas ejecutar `math.sin(matriz)` usando el módulo estándar `math` de Python, obtendrás inmediatamente `TypeError: only length-1 arrays can be converted to Python scalars`. Las funciones del módulo `math` solo aceptan números individuales; para arreglos utiliza siempre sus equivalentes en `np`.
:::

:::{note} Restricción de Dominio Matemático en `ufuncs`
* `np.sin`, `np.cos` y `np.tan` esperan ángulos en **radianes**. Pasar grados directamente (ej. `np.sin(90)`) calculará el seno de 90 radianes (`0.8939`), no de $90^\circ$ (`1.0`).
* Calcular `np.sqrt(-4)` o `np.log(0)` devolverá `nan` o `-inf` respectivamente sin detener el flujo de ejecución.
:::

### 🧩 Micro-reto 4.3
1. Crea un vector `x` con **11 valores equidistantes** entre `0` y `2 * np.pi` usando `np.linspace()`.
2. Verifica numéricamente la identidad trigonométrica fundamental $\sin^2(x) + \cos^2(x) = 1$ calculando `identidad = np.square(np.sin(x)) + np.square(np.cos(x))` e imprime el vector resultante.
3. Dada la matriz `M = np.arange(1, 13).reshape(3, 4)`, eleva cada elemento a la potencia `3` usando `np.power(M, 3)` y calcula luego la raíz cuadrada (`np.sqrt()`) del resultado.

In [8]:
# Escribe tu solución aquí:

## ✅ Resumen del Módulo 4
* Las operaciones aritméticas básicas (`+`, `-`, `*`, `/`, `**`) en NumPy son **vectorizadas** y se aplican celda por celda sin necesidad de ciclos `for`.
* El **Broadcasting** alinea las dimensiones de dos arreglos **de derecha a izquierda**, permitiendo que cualquier eje de tamaño `1` se estire virtualmente para coincidir con el eje del otro arreglo.
* Para operar una matriz `(M, N)` con un vector columna de tamaño `M`, transforma el vector a forma `(M, 1)` mediante `vec[:, np.newaxis]`.
* Utiliza siempre las `ufuncs` de NumPy (`np.sqrt`, `np.log`, `np.sin`, `np.power`) en lugar de la librería `math` estándar de Python.